<a href="https://colab.research.google.com/github/joyngeno21-ui/Joy-AI-Assignment/blob/main/Notebooks/Chap07/7_3_Initialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 7.3: Initialization**

This notebook explores weight initialization in deep neural networks as described in section 7.5 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

First let's define a neural network.  We'll just choose the weights and biases randomly for now

In [2]:
def init_params(K, D, sigma_sq_omega):
  # Set seed so we always get the same random numbers
  np.random.seed(0)

  # Input layer
  D_i = 1
  # Output layer
  D_o = 1

  # Make empty lists
  all_weights = [None] * (K+1)
  all_biases = [None] * (K+1)

  # Create input and output layers
  all_weights[0] = np.random.normal(size=(D, D_i))*np.sqrt(sigma_sq_omega)
  all_weights[-1] = np.random.normal(size=(D_o, D)) * np.sqrt(sigma_sq_omega)
  all_biases[0] = np.zeros((D,1))
  all_biases[-1]= np.zeros((D_o,1))

  # Create intermediate layers
  for layer in range(1,K):
    all_weights[layer] = np.random.normal(size=(D,D))*np.sqrt(sigma_sq_omega)
    all_biases[layer] = np.zeros((D,1))

  return all_weights, all_biases

In [3]:
# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation

In [4]:
def compute_network_output(net_input, all_weights, all_biases):

  # Retrieve number of layers
  K = len(all_weights)-1

  # We'll store the pre-activations at each layer in a list "all_f"
  # and the activations in a second list "all_h".
  all_f = [None] * (K+1)
  all_h = [None] * (K+1)

  #For convenience, we'll set
  # all_h[0] to be the input, and all_f[K] will be the output
  all_h[0] = net_input

  # Run through the layers, calculating all_f[0...K-1] and all_h[1...K]
  for layer in range(K):
      # Update preactivations and activations at this layer according to eqn 7.5
      all_f[layer] = all_biases[layer] + np.matmul(all_weights[layer], all_h[layer])
      all_h[layer+1] = ReLU(all_f[layer])

  # Compute the output from the last hidden layer
  all_f[K] = all_biases[K] + np.matmul(all_weights[K], all_h[K])

  # Retrieve the output
  net_output = all_f[K]

  return net_output, all_f, all_h

Now let's investigate how the size of the outputs vary as we change the initialization variance:


In [5]:
# Number of layers
K = 5
# Number of neurons per layer
D = 8
# Input layer
D_i = 1
# Output layer
D_o = 1
# Set variance of initial weights to 1
sigma_sq_omega = 1.0
# Initialize parameters
all_weights, all_biases = init_params(K,D,sigma_sq_omega)

n_data = 1000
data_in = np.random.normal(size=(1,n_data))
net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)

for layer in range(1,K+1):
  print("Layer %d, std of hidden units = %3.3f"%(layer, np.std(all_h[layer])))

Layer 1, std of hidden units = 0.811
Layer 2, std of hidden units = 1.472
Layer 3, std of hidden units = 4.547
Layer 4, std of hidden units = 8.896
Layer 5, std of hidden units = 10.106


In [10]:
K = 50
D = 80
D_i = 1
D_o = 1

# Experiment with different sigma_sq_omega values
print("\n--- Forward Pass Variance Experiment ---")
for sigma_sq_omega in [1.0, 2.0/D, 2.0/D_i]: # Typical values for He initialization
  print(f"\nSigma_sq_omega = {sigma_sq_omega:.4f}")
  all_weights, all_biases = init_params(K,D,sigma_sq_omega)

  n_data = 1000
  data_in = np.random.normal(size=(1,n_data))
  net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)

  for layer in range(1,K+1):
    if layer % 10 == 0 or layer == K or layer == 1: # Print for some layers to avoid too much output
      print("Layer %d, std of hidden units = %3.3f"%(layer, np.std(all_h[layer])))



--- Forward Pass Variance Experiment ---

Sigma_sq_omega = 1.0000
Layer 1, std of hidden units = 0.622
Layer 10, std of hidden units = 7366234.399
Layer 20, std of hidden units = 999886829483868.875
Layer 30, std of hidden units = 127571735900692099891200.000
Layer 40, std of hidden units = 6426783972194130038049969537024.000
Layer 50, std of hidden units = 753218502856424628936041819469258948608.000

Sigma_sq_omega = 0.0250
Layer 1, std of hidden units = 0.098
Layer 10, std of hidden units = 0.072
Layer 20, std of hidden units = 0.095
Layer 30, std of hidden units = 0.119
Layer 40, std of hidden units = 0.058
Layer 50, std of hidden units = 0.067

Sigma_sq_omega = 2.0000
Layer 1, std of hidden units = 0.880
Layer 10, std of hidden units = 235719500.767
Layer 20, std of hidden units = 1023884113391483264.000
Layer 30, std of hidden units = 4180270641993886425816236032.000
Layer 40, std of hidden units = 6738971430427449807652595618424029184.000
Layer 50, std of hidden units = 25273819

Now let's define a loss function.  We'll just use the least squares loss function. We'll also write a function to compute dloss_doutput


In [7]:
def least_squares_loss(net_output, y):
  return np.sum((net_output-y) * (net_output-y))

def d_loss_d_output(net_output, y):
    return 2*(net_output -y);

Here's the code for the backward pass

In [8]:
# We'll need the indicator function
def indicator_function(x):
  x_in = np.array(x)
  x_in[x_in>=0] = 1
  x_in[x_in<0] = 0
  return x_in

# Main backward pass routine
def backward_pass(all_weights, all_biases, all_f, all_h, y):
  # Retrieve number of layers
  K = len(all_weights) - 1

  # We'll store the derivatives dl_dweights and dl_dbiases in lists as well
  all_dl_dweights = [None] * (K+1)
  all_dl_dbiases = [None] * (K+1)
  # And we'll store the derivatives of the loss with respect to the activation and preactivations in lists
  all_dl_df = [None] * (K+1)
  all_dl_dh = [None] * (K+1)
  # Again for convenience we'll stick with the convention that all_h[0] is the net input and all_f[k] in the net output

  # Compute derivatives of net output with respect to loss
  all_dl_df[K] = np.array(d_loss_d_output(all_f[K],y))

  # Now work backwards through the network
  for layer in range(K,-1,-1):
    # Calculate the derivatives of biases at layer from all_dl_df[K]. (eq 7.13, line 1)
    all_dl_dbiases[layer] = np.array(all_dl_df[layer])
    # Calculate the derivatives of weight at layer from all_dl_df[K] and all_h[K] (eq 7.13, line 2)
    all_dl_dweights[layer] = np.matmul(all_dl_df[layer], all_h[layer].transpose())

    # Calculate the derivatives of activations from weight and derivatives of next preactivations (eq 7.13, line 3 second part)
    all_dl_dh[layer] = np.matmul(all_weights[layer].transpose(), all_dl_df[layer])
    # Calculate the derivatives of the pre-activation f with respect to activation h (eq 7.13, line 3, first part)
    if layer > 0:
      all_dl_df[layer-1] = indicator_function(all_f[layer-1]) * all_dl_dh[layer]

  return all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df

Now let's look at what happens to the magnitude of the gradients on the way back.

In [9]:
# Number of layers
K = 5
# Number of neurons per layer
D = 8
# Input layer
D_i = 1
# Output layer
D_o = 1
# Set variance of initial weights to 1
sigma_sq_omega = 1.0
# Initialize parameters
all_weights, all_biases = init_params(K,D,sigma_sq_omega)

# For simplicity we'll just consider the gradients of the weights and biases between the first and last hidden layer
n_data = 100
aggregate_dl_df = [None] * (K+1)
for layer in range(1,K):
  # These 3D arrays will store the gradients for every data point
  aggregate_dl_df[layer] = np.zeros((D,n_data))


# We'll have to compute the derivatives of the parameters for each data point separately
for c_data in range(n_data):
  data_in = np.random.normal(size=(1,1))
  y = np.zeros((1,1))
  net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)
  all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df = backward_pass(all_weights, all_biases, all_f, all_h, y)
  for layer in range(1,K):
    aggregate_dl_df[layer][:,c_data] = np.squeeze(all_dl_df[layer])

for layer in reversed(range(1,K)):
  print("Layer %d, std of dl_dh = %3.3f"%(layer, np.std(aggregate_dl_df[layer].ravel())))


Layer 4, std of dl_dh = 56.472
Layer 3, std of dl_dh = 109.132
Layer 2, std of dl_dh = 340.657
Layer 1, std of dl_dh = 446.654


In [12]:
K = 50
D = 80
D_i = 1
D_o = 1

# Experiment with different sigma_sq_omega values
print("\n--- Backward Pass Variance Experiment ---")
for sigma_sq_omega in [1.0, 2.0/D, 2.0/D_i]: # Typical values for He initialization
  print(f"\nSigma_sq_omega = {sigma_sq_omega:.4f}")
  all_weights, all_biases = init_params(K,D,sigma_sq_omega)

  n_data = 100
  aggregate_dl_df = [None] * (K+1)
  for layer in range(1,K):
    aggregate_dl_df[layer] = np.zeros((D,n_data))

  for c_data in range(n_data):
    data_in = np.random.normal(size=(1,1))
    y = np.zeros((1,1))
    net_output, all_f, all_h = compute_network_output(data_in, all_weights, all_biases)
    all_dl_dweights, all_dl_dbiases, all_dl_dh, all_dl_df = backward_pass(all_weights, all_biases, all_f, all_h, y)
    for layer in range(1,K):
      aggregate_dl_df[layer][:,c_data] = np.squeeze(all_dl_df[layer])

  for layer in reversed(range(1,K)):
    if layer % 10 == 0 or layer == K-1 or layer == 1: # Print for some layers to avoid too much output
      print("Layer %d, std of dl_dh = %3.3f"%(layer, np.std(aggregate_dl_df[layer].ravel())))



--- Backward Pass Variance Experiment ---

Sigma_sq_omega = 1.0000
Layer 49, std of dl_dh = 18817849764518896023941902245604242751488.000
Layer 40, std of dl_dh = 258241445087590735643133194351627983599477719040.000
Layer 30, std of dl_dh = 27962097471983042838261328844134491112183619548352610304.000
Layer 20, std of dl_dh = 2521931090077676164675708087308560164602781204488721344485130240.000
Layer 10, std of dl_dh = 291677266860870437780232576190188443241514488697868118675201045020278784.000
Layer 1, std of dl_dh = 3864161615668244381461267510373904423054174106385627368088294218476908925943808.000

Sigma_sq_omega = 0.0250
Layer 49, std of dl_dh = 0.042
Layer 40, std of dl_dh = 0.035
Layer 30, std of dl_dh = 0.037
Layer 20, std of dl_dh = 0.033
Layer 10, std of dl_dh = 0.037
Layer 1, std of dl_dh = 0.030

Sigma_sq_omega = 2.0000
Layer 49, std of dl_dh = 1262844520619534837442055421248997905660861480960.000
Layer 40, std of dl_dh = 392139698922776982294452606829021737588933712355075817